In [0]:
from pyspark.sql.functions import current_timestamp, lit, input_file_name
BUCKET="capstone-ecomm-pawan"
S3 = f"s3a://{BUCKET}"

try:
    batch_number = dbutils.widgets.get("batch_number")
except:
    dbutils.widgets.text("batch_number", "1")
    batch_number = "1"

print(f"Running Bronze ingestion for batch: {batch_number}")




In [0]:
def ingest_to_bronze(s3_path, table_name, mode="append"):
    """Read CSV from S3, add audit columns, save as Delta."""
    print(f"\n  Loading: bronze.{table_name} from {s3_path}")
    
    try:
        df = spark.read \
            .option("header", True) \
            .option("inferSchema", True) \
            .csv(s3_path)
    except Exception as e:
        print(f"  ERROR reading {s3_path}: {e}")
        return 0
    
    df = df.withColumn("ingestion_timestamp", current_timestamp()) \
           .withColumn("source_file", lit(s3_path.split("/")[-1])) \
           .withColumn("batch_id", lit(f"batch_{batch_number}"))
    
    row_count = df.count()
    df.write.format("delta").mode(mode).saveAsTable(f"bronze.{table_name}")
    print(f"  Result: bronze.{table_name} — {row_count:,} rows ({mode})")
    return row_count


In [0]:
total_rows = 0

# BATCH 1: ALL dims (overwrite) + 25% facts (overwrite)
if batch_number == "1":
    print("\n" + "="*60)
    print("BATCH 1: Dimensions (full) + Facts (25%)")
    print("="*60)
    
    # Dimensions — overwrite (first and only load)
    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/product_category_name_translation.csv",
        "category_translation", "overwrite")
    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/geolocation_dataset.csv",
        "geolocation", "overwrite")
    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/sellers_dataset.csv",
        "sellers", "overwrite")
    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/customers_dataset.csv",
        "customers", "overwrite")
    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/products_dataset.csv",
        "products", "overwrite")
    
    # Facts — overwrite (first load)
    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/orders_dataset.csv",
        "orders", "overwrite")
    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/order_items_dataset.csv",
        "order_items", "overwrite")
    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/order_payments_dataset.csv",
        "order_payments", "overwrite")
    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/order_reviews_dataset.csv",
        "order_reviews", "overwrite")


# BATCH 2-4: Facts only (append — incremental) 
elif batch_number in ["2", "3", "4"]:
    print(f"\n{'='*60}")
    print(f"BATCH {batch_number}: Facts only (incremental append)")
    print("="*60)
    
    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_{batch_number}/orders_dataset.csv",
        "orders", "append")
    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_{batch_number}/order_items_dataset.csv",
        "order_items", "append")
    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_{batch_number}/order_payments_dataset.csv",
        "order_payments", "append")
    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_{batch_number}/order_reviews_dataset.csv",
        "order_reviews", "append")


print(f"\n{'='*60}")
print(f"Bronze ingestion COMPLETE for batch {batch_number}")
print(f"Total rows ingested this run: {total_rows:,}")
print("="*60)

In [0]:
%sql
SELECT * FROM bronze.orders LIMIT 10;